# Taller 3: Exploratory Data Analysis (EDA) y Data Wrangling

## Introducción

Instacart es una plataforma de entrega de comestibles donde los clientes pueden hacer un pedido y recibirlo, de manera similar a como funcionan Uber Eats y Door Dash.
El conjunto de datos que se te ha proporcionado ha sido modificado del original. Hemos reducido el tamaño del conjunto para que tus cálculos se ejecuten más rápido y hemos introducido deliberadamente valores ausentes (missing values) y duplicados. Sin embargo, hemos conservado las distribuciones poblacionales de los datos originales.

**Tu misión profesional:** Como Científico de Datos, aplicarás metodológicamente las 4 dimensiones de la calidad de los datos para auditar tu dataset. Luego, realizarás un análisis exploratorio empleando medidas de tendencia central, variabilidad, diagramas de dispersión, y responderás preguntas de negocio tras construir una tabla desnormalizada (One Big Table - OBT).

> **Aviso Importante:** Para cada paso, escribe introducciones donde expongas qué harás, bloques intermedios justificando el porqué de tus decisiones y tu limpieza estadística, y cierra siempre con una conclusión de lo hallado.

## Diccionario de Datos

Hay cinco tablas en el conjunto de datos que deberás usar en conjunto:

- `instacart_orders.csv`: cada fila es un pedido en la app de Instacart
    - `'order_id'`: ID que identifica de manera única cada pedido
    - `'user_id'`: ID que identifica de manera única a cada cliente
    - `'order_number'`: el número de veces que este cliente ha hecho un pedido
    - `'order_dow'`: día de la semana en que se hizo el pedido (0 es domingo)
    - `'order_hour_of_day'`: hora del día en que se hizo el pedido
    - `'days_since_prior_order'`: número de días desde que este cliente hizo su último pedido
- `products.csv`: cada fila corresponde a un producto
    - `'product_id'`: ID único del producto
    - `'product_name'`: nombre del producto
    - `'aisle_id'`: ID único de la categoría del pasillo
    - `'department_id'`: ID único de la categoría del departamento
- `order_products.csv`: cada fila corresponde a un artículo en un pedido
    - `'order_id'`: ID único del pedido
    - `'product_id'`: ID único del producto
    - `'add_to_cart_order'`: el orden secuencial en el que se añadió cada artículo al carrito
    - `'reordered'`: 0 si el cliente no había pedido este producto antes, 1 si ya lo había hecho
- `aisles.csv`
    - `'aisle_id'`: ID único del pasillo
    - `'aisle'`: nombre del pasillo
- `departments.csv`
    - `'department_id'`: ID único del departamento
    - `'department'`: nombre del departamento

# Paso 1: Exploración Inicial

Lee los archivos ubicados en el directorio `/datos/` (`instacart_orders.csv`, `products.csv`, `aisles.csv`, `departments.csv` y `order_products.csv`) fijándote que algunos están separados por `;` u otros delimitadores. 

Tras leerlos, ejecuta funciones de inspección básica (`head()`, `info()`) y comprueba una muestra aleatoria (Muestreo Aleatorio Simple usando `sample()`) de cada tabla para evitar el *sesgo de posición*. Verifica si los tipos computacionales coinciden con su rol estadístico esperado.

In [1]:
# Importación de librerias y modulos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Carga de archivos de datos
orders=pd.read_csv('./datos/instacart_orders.csv', sep=';')
products=pd.read_csv('./datos/products.csv', sep=';')
aisles=pd.read_csv('./datos/aisles.csv', sep=';')
departments=pd.read_csv('./datos/departments.csv', sep=';')
order_products=pd.read_csv('./datos/order_products.csv', sep=';')

In [3]:
# Iteración sobre los dataframes para mostrar información relevante
for nombre, df in [('orders', orders), ('products', products),
                   ('aisles', aisles), ('departments', departments),
                   ('order_products', order_products)]:

    print(f"TABLA: {nombre}\n")
    print(f"Encabezado de la tabla {nombre}:\n")
    print(df.head())
    print()          
    print(f"Información de la tabla {nombre}:\n")
    print(df.info())     
    print()     
    print(f"Muestra aleatoria de la tabla {nombre}:\n")
    print(df.sample(20))        
    print()
    print(f"Dimensiones de la tabla {nombre}:")
    print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
    print()

TABLA: orders

Encabezado de la tabla orders:

   order_id  user_id  order_number  order_dow  order_hour_of_day  \
0   1515936   183418            11          6                 13   
1   1690866   163593             5          5                 12   
2   1454967    39980             4          5                 19   
3   1768857    82516            56          0                 20   
4   3007858   196724             2          4                 12   

   days_since_prior_order  
0                    30.0  
1                     9.0  
2                     2.0  
3                    10.0  
4                    17.0  

Información de la tabla orders:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_numbe

### Escribe tus hallazgos iniciales aquí:
(Escribe textualmente qué dimensiones, tipos de datos ves incoherentes y qué opinas estadísticamente de la primera radiografía de los datos).

Las dimensiones de los datos son los siguientes:

- orders: 478967 filas × 6 columnas

- products: 49694 filas × 4 columnas
    
- aisles: 134 filas × 2 columnas
    
- departments: 21 filas × 2 columnas
    
- order_products: 4545007 filas × 4 columnas 

Los dataset que presentan alguna incoherencia en su tipo de datos son:
    
- orders: *days_since_prior_order* debería ser entero (int64) y en este caso es float64, porque los días son números enteros
    
- order_products: *add_to_cart_order* es float64, cuando por la naturaleza de esa variable sería conveniente que sea un entero (int64)

Con respecto a un argumento estadístico de estos primeros hallazgos, las tablas **aisles** y **departments** son pequeñas y están limpias, denotando un control sobre estas por la poca cantidad de datos. Por otro lado, el resto de tablas presentan una cantidad de datos más robusta para un análiis estadístico así como detección de tendencias, patrones y distribuciones que colaboren en la toma de desiciones basadas en datos

# Paso 2: Auditoría de Calidad de Datos (4 Dimensiones)

Basado en la teoría vista, somete a tus 5 tablas al escrutinio en las cuatro dimensiones.

## 2.1 Precisión: Identificación y limpieza de duplicados explícitos e implícitos
Encuentra y elimina duplicados explícitos en la tabla de `orders`. Nota si se agrupan por algún motivo (¿Son de un día y hora en particular?). Asimismo, busca duplicados por ID de producto (tricky duplicates) o nombres duplicados con problemas de mayúsculas y minúsculas.

In [4]:
# Ver la cantida de duplicados explícitos en orders
dupl_orders = orders[orders.duplicated()]
print(f"Duplicados explícitos en orders: {len(dupl_orders)}")

# Ver los duplicados 
display(dupl_orders.head(15))

Duplicados explícitos en orders: 15


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
145574,794638,50898,24,3,2,2.0
223105,2160484,107525,16,3,2,30.0
230807,1918001,188546,14,3,2,16.0
266232,1782114,106752,1,3,2,NaN
273805,1112182,202304,84,3,2,6.0
284038,2845099,31189,11,3,2,7.0
311713,1021560,53767,3,3,2,9.0
321100,408114,68324,4,3,2,18.0
323900,1919531,191501,32,3,2,7.0
345917,2232988,82565,1,3,2,NaN


Se aprecia que los datos que tienen duplicados tienen un patrón, suceden los días Miércoles (Día 3) a las 2 am. Por lo podría suponerse que existe un bug en el servidor que toma los datos a esa hora.

Como se ve que tienen un patrón, una decisión inteligente sería **NO BORRARLOS** ya que presentan información útil para un mejoramiento en el sistema, ya con este hallazgo, el departamento de datos de la empresa podría verificar que está pasando y estudiar de mejor manera el caso. Si se los borra, se perdería esa información y el sistema de la aplicación seguiria con el problema.

In [5]:
# Verificar el ID y nombres en products

# Creación de una columna para comparar los nombres 
products['product_name_lower'] = products['product_name'].str.lower().str.strip()

# Ver los pares duplicados en product_name_lower
dupl_prod = products[products.duplicated(subset='product_name_lower', keep=False) &
                     products['product_name_lower'].notna()] # Se excluyen los nulos

display(
    dupl_prod[['product_id', 'product_name', 'product_name_lower']]
    .sort_values('product_name_lower')
    .reset_index(drop=True)
    .style.set_caption(f"Productos duplicados (case-insensitive): {len(dupl_prod)}")
)

,product_id,product_name,product_name_lower
0,23340,18-in-1 Hemp Peppermint Pure-Castile Soap,18-in-1 hemp peppermint pure-castile soap
1,31845,18-In-1 Hemp Peppermint Pure-Castile Soap,18-in-1 hemp peppermint pure-castile soap
2,19942,Aged Balsamic Vinegar Of Modena,aged balsamic vinegar of modena
3,13153,Aged Balsamic Vinegar of Modena,aged balsamic vinegar of modena
4,24831,Albacore Solid White Tuna in Water,albacore solid white tuna in water
5,22583,Albacore Solid White Tuna In Water,albacore solid white tuna in water
6,9038,American Cheese slices,american cheese slices
7,516,American Cheese Slices,american cheese slices
8,49531,Anchovy Fillets In Olive Oil,anchovy fillets in olive oil
9,12326,Anchovy Fillets in Olive Oil,anchovy fillets in olive oil


In [6]:
# Duplicados por ID
dupl_prod_id = products[products.duplicated(subset='product_id', keep=False)]

display(
    dupl_prod_id[['product_id', 'product_name', 'product_name_lower']]
    .sort_values('product_id')
    .reset_index(drop=True)
    .style.set_caption(f"Duplicados por product_id: {len(dupl_prod_id)}")
)

,product_id,product_name,product_name_lower


Como se encuentran duplicados por product_name, se hará una homogenización de estos, y se conservará solo los nombres con el menor product_id

Ya que en teoria es el primero que se ingresó al sistema, por lo tanto sería el product_id original.

In [7]:
products = (products
            .sort_values('product_id')                              # menor ID primero
            .drop_duplicates(subset='product_name_lower', keep='first')  # conserva el primero
            .drop(columns='product_name_lower')
            .reset_index(drop=True))

display(
    products.head(10)
    .style.set_caption(f"Products tras normalización — shape: {products.shape}")
)

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
4,5,Green Chile Anytime Sauce,5,13
5,6,Dry Nose Oil,11,11
6,7,Pure Coconut Water With Orange,98,7
7,8,Cut Russet Potatoes Steam N' Mash,116,1
8,9,Light Strawberry Blueberry Yogurt,120,16
9,10,Sparkling Orange Juice & Prickly Pear Beverage,115,7


In [8]:
# Verificar si hay nombres duplicados

products['product_name_lower'] = products['product_name'].str.lower().str.strip()

# Ver los pares duplicados en product_name_lower
dupl_prod = products[products.duplicated(subset='product_name_lower', keep=False) &
                     products['product_name_lower'].notna()] # Se excluyen los nulos

display(
    dupl_prod[['product_id', 'product_name', 'product_name_lower']]
    .sort_values('product_name_lower')
    .reset_index(drop=True)
    .style.set_caption(f"Productos duplicados (case-insensitive): {len(dupl_prod)}")
)

,product_id,product_name,product_name_lower


También se eliminarán nulos en products, ya que no presentan información útil, no hay forma de identificar el producto que es

In [9]:
products = products.dropna(subset=['product_name']).reset_index(drop=True)

display(
    products.head(10)
    .style.set_caption(f"Products tras eliminar NaN — shape: {products.shape}")
)

,product_id,product_name,aisle_id,department_id,product_name_lower
0,1,Chocolate Sandwich Cookies,61,19,chocolate sandwich cookies
1,2,All-Seasons Salt,104,13,all-seasons salt
2,3,Robust Golden Unsweetened Oolong Tea,94,7,robust golden unsweetened oolong tea
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1,smart ones classic favorites mini rigatoni with vodka cream sauce
4,5,Green Chile Anytime Sauce,5,13,green chile anytime sauce
5,6,Dry Nose Oil,11,11,dry nose oil
6,7,Pure Coconut Water With Orange,98,7,pure coconut water with orange
7,8,Cut Russet Potatoes Steam N' Mash,116,1,cut russet potatoes steam n' mash
8,9,Light Strawberry Blueberry Yogurt,120,16,light strawberry blueberry yogurt
9,10,Sparkling Orange Juice & Prickly Pear Beverage,115,7,sparkling orange juice & prickly pear beverage


### Conclusiones sobre la Precisión:
Explica por qué eliminaste (o consolidaste) la información y qué nos enseña sobre la ingesta del sistema.

Los duplicados en orders se conservan deliberadamente. Al seguir un patrón no aleatorio (miércoles a las 2 AM), su eliminación implicaría pérdida de información potencialmente útil para auditoría del pipeline.

Los nombres de los productos duplicados se homogenizaron y se conservó el id menor, teniendo en cuenta que sería el valor canónico. Ya con eso, se establece una línea base para tener en cuenta que estos valores son los reales, para homogenizar la base de datos

## 2.2 Completitud: Tratamiento de Valores Ausentes
Navega por las tablas en busca de NaNs. Es crítico que analices si el patrón de valores ausentes se trata de MCAR (Missing Completely at Random), MAR o MNAR.

1. Revisa nombres de productos vacíos.
2. Revisa NaNs en `days_since_prior_order`. (¿Son clientes haciendo su primera orden?)
3. Revisa `add_to_cart_order` en `order_products`. (¿Qué pasa después de 64 artículos?)

Decide si debes imputar por tendencia central, usar ML, imputar un valor indicador ficticio (e.g. -1 o 'Desconocido') o eliminar las filas. ¡Aplica tus soluciones!

### Conclusiones sobre la Completitud:
Argumenta qué mecanismo de ausencia (Rubin) encontraste en cada escenario y por qué tus estrategias de imputación no generaron sesgos severos.

## 2.3 Sensibilidad: Tratamiento de Outliers (Valores Atípicos)
Centra tu atención en la tabla de órdenes, específicamente en `days_since_prior_order` o conteos agrupados. Usando gráficos estadísticos (Boxplots) y computando el Rango Intercuartílico (IQR / Regla de Tukey) o Z-Scores, encuentra y reporta valores atípicos. ¿Existen usuarios que compran muy fuera del rango promedio de frecuencia?

### Conclusiones sobre la Sensibilidad:
Decide y justifica la estrategia que usaste para estos casos (eliminar, mantener como outlier real o winsorizar).

# Paso 3: Data Wrangling & Construcción de One Big Table (OBT)

Para poder responder preguntas de negocio de manera eficiente y aplicando correlaciones y cruces, un científico de datos típicamente convierte o "aplana" las tablas normalizadas operacionales en **One Big Table (OBT)**. La OBT agrupa (hace merges) de las distintas dimensiones transaccionales alrededor del concepto central (en este caso el *Articulo pedido por Orden*), lo que nos da una vista panorámica (desnormalización estructural) para el análisis.

**Tu tarea:** Une las cinco tablas previamente purgadas formando un solo DataFrame analítico. Te recomendamos ir uniendo ordenes y el detalle, luego producto, pasillo y finalmente departamento.

# Paso 4: Análisis Multivariado y de Negocio

Con tu conjunto de datos ahora limpio (*trusted/gold data*) mediante la auditoría y unificado en tu OBT, usa tus conocimientos de agregación y visualización estadística para responder las siguientes interrogantes. Usa histogramas, KDE y/o gráficos de dispersión donde la distribución lo merezca.

## [A] Preguntas Esenciales

**A1. La hora y el día: Verificación del dominio:**
Verifica con código que `order_hour_of_day` y `order_dow` tienen distribuciones lógicas basándonos en tu conocimiento del mundo real. Construye histogramas para ver qué picos de horas concentran compras. Aplica gráficos de barras para evaluar diferencias de días y horas (comparemos, por ejemplo, distribuciones de la demanda los Miércoles vs. Sábados). Verifica si existe una diferencia de horas por dia.

**A2. Distribuciones temporales de recompra:**
¿Cuánto tiempo transcurre estadísticamente para que alguien vuelva a realizar otra orden? Grafica este tiempo de espera y concluye sobre la concentración de los datos (¿asimetría?).

### Tira aquí tus hallazgos estadísticos para el grupo A:
(Debes hablar de cómo se distribuye la demanda, tendencias centrales encontradas y asimetrías de cola).

## [B] Profundización (Segmentación Categórica)

**B1. Retención y recurrencia (Número de órdenes por usuario):**
Aislando o agrupando a nivel de granularidad de cliente, describe la variabilidad y distribución de compras que hacen. ¿Existen colas largas de clientes extra-leales?

**B2. Productos Top: El principio de Pareto:**
Genera un top 20 de los productos más solicitados globalmente. Para cada producto reporta su ratio de recompra. ¿Algunos productos tienen una correlación fuerte entre ser de un 'pasillo' y volverse recompras aseguradas?

### Conclusiones sobre el perfil del carrito [Grupo B]:

## [C] Patrones de Causalidad y Exposición (Hard)

**C1. El tamaño de la canasta comercial:**
¿Cuántos artículos en promedio compran las personas estadísticamente? Aplica gráficos sobre esta proporción y analiza la curtosis (grado de acumulación) de esta distribución del tamaño del pedido.

**C2. ¿Son los primeros productos un síntoma de fidelidad? (Spearman o Pearson):**
Para los 20 objetos que con mayor frecuencia las personas agregan como posición #1 al carrito, elabora una lógica que calcule una matriz de correlación (en variables generadas o agrupadas) o genera gráficos de dispersión (Scatterplot). Compara si, a nivel de producto, ser posicionado primero tiene relación lineal con ser un producto de constante recompra global (el campo 'reordered').

**C3. Para cada producto, ¿cual es la proporcion de re-compras?**

**C4. Para cada usuario, ¿cual es la proporcion de re-compras de los productos comprados?**

**C5. ¿Cuales son los top 20 productos que los clientes ponen primero en sus carritos?**

### Tus conclusiones avanzadas [Grupo C]:
(Enuncia las posibles relaciones matemáticas encontradas o variables dispersas)

# Conclusión General Ejecutiva
Resume en un párrafo las principales fortalezas, falencias detectadas en la gobernanza inicial de este dataset, y los descubrimientos de negocio clave que proporcionarías a la directiva de Instacart.